In [1]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

# SMOTE split for supervised models (LR, RF)
X_train_smote, y_train_smote = joblib.load('../data/train_smote.joblib')
X_test, y_test = joblib.load('../data/test.joblib')

# Rebuild pre-SMOTE split for Isolation Forest (unsupervised)
df = pd.read_csv('../data/creditcard.csv')
scaler = joblib.load('../backend/models/scaler.joblib')
scaled = scaler.transform(df[['Amount', 'Time']])
df['Amount_scaled'] = scaled[:, 0]
df['Time_scaled'] = scaled[:, 1]
df.drop(columns=['Amount', 'Time'], inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

X_train_pre_smote, _, y_train_pre_smote, _ = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'IsolationForest train split (pre-SMOTE): {X_train_pre_smote.shape}')
print(f'LR/RF train split (SMOTE): {X_train_smote.shape}')
print(f'Test split: {X_test.shape}')

IsolationForest train split (pre-SMOTE): (227845, 30)
LR/RF train split (SMOTE): (454902, 30)
Test split: (56962, 30)


In [2]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=200,
    contamination=0.002,   # ~0.17% fraud rate
    random_state=42,
    n_jobs=-1
)

# Train on pre-SMOTE data (unsupervised — labels irrelevant)
iso.fit(X_train_pre_smote)

# Predictions: -1 = anomaly (fraud), 1 = normal
iso_preds_raw = iso.predict(X_test)
iso_preds = (iso_preds_raw == -1).astype(int)  # convert to 0/1

joblib.dump(iso, '../backend/models/isolation_forest.joblib')

['../backend/models/isolation_forest.joblib']

In [3]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    class_weight='balanced',  # compensates for imbalance even without SMOTE
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)

lr.fit(X_train_smote, y_train_smote)
lr_probs = lr.predict_proba(X_test)[:, 1]   # probability of fraud

joblib.dump(lr, '../backend/models/logistic_regression.joblib')

['../backend/models/logistic_regression.joblib']

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_smote, y_train_smote)
rf_probs = rf.predict_proba(X_test)[:, 1]

joblib.dump(rf, '../backend/models/random_forest.joblib')

['../backend/models/random_forest.joblib']

In [5]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores_lr = [f1_score(y_test, (lr_probs >= t).astype(int)) for t in thresholds]
f1_scores_rf = [f1_score(y_test, (rf_probs >= t).astype(int)) for t in thresholds]

best_threshold_lr = thresholds[np.argmax(f1_scores_lr)]
best_threshold_rf = thresholds[np.argmax(f1_scores_rf)]

print(f"Best threshold LR: {best_threshold_lr:.2f}, RF: {best_threshold_rf:.2f}")
joblib.dump({'lr': best_threshold_lr, 'rf': best_threshold_rf}, '../backend/models/thresholds.joblib')

Best threshold LR: 0.89, RF: 0.80


['../backend/models/thresholds.joblib']